# Demo: OWL 2 RL Inference

This notebook demonstrates the current end-to-end OWL 2 RL inference path at the RDFLib level:

- add asserted triples to an RDFLib `Dataset`
- let `RETEStore` materialize inferred triples through `RETEEngine`
- confirm that the inferred goal triple is present in the graph
- inspect the rule applications that justify that entailment

It is the OWL 2 RL counterpart to [demo-rdfs-inference.ipynb](./demo-rdfs-inference.ipynb). The call shape is the same (`specification.dataset()`); only the ruleset and vocabulary differ.

## Where does this notebook fit?

| Topic | Notebook |
|-------|----------|
| RDFS inference + derivation inspection | [demo-rdfs-inference.ipynb](./demo-rdfs-inference.ipynb) |
| OWL 2 RL inference + derivation inspection | **this notebook** |
| RDFS inference retraction after graph updates | [demo-rdfs-retraction.ipynb](./demo-rdfs-retraction.ipynb) |
| Contradiction diagnostics + explanation | [demo-contradiction-rules.ipynb](./demo-contradiction-rules.ipynb) |

## 1. Inference-Backed RDFLib Graph

We use an in-memory RDFLib store wrapped by `RETEStore` so that ordinary graph updates trigger OWL 2 RL materialization.
An `InMemoryDerivationLogger` is enabled as well so we can inspect which rule applications justify one inferred triple.

> ℹ️: You don't need a `derivation_logger` to use OWL 2 RL inference.
> If you don't need explanations, just don't override `derivation_logger_factory`.

`OWL2_RL_RETE_MATERIALIZED` includes both OWL 2 RL entailment rules and contradiction diagnostics. The default recorder policy raises when a contradiction is detected; this tutorial stays in the positive-entailment path.

In [1]:
from dataclasses import replace

from rdflib import Dataset, Graph, URIRef
from rdflib_reasoning.engine import (
    OWL2_RL_RETE_MATERIALIZED,
    InMemoryDerivationLogger,
)

spec = replace(
    OWL2_RL_RETE_MATERIALIZED,
    derivation_logger_factory=lambda _context: InMemoryDerivationLogger(),
)

dataset: Dataset = spec.dataset()
graph: Graph = dataset.default_graph

## 2. Triggering Inference

We create some basic terms and construct a minimal ontology of 3 statements:

1. `ex:Person` is declared equivalent to `ex:Human`
2. `ex:Human` is a subclass of `ex:Animal`
3. `ex:alice` is asserted to be a `ex:Person`

Due to our forward-chaining inference engine, the OWL 2 RL profile materializes a visible closure beyond those asserted triples. In particular, class equivalence propagates typing (`cax-eqc1`), and subclass typing propagates membership (`cax-sco`). Schema rules such as `scm-eqc2` may also expand `owl:equivalentClass` into mutual `rdfs:subClassOf` links.

Compared with the RDFS demo, the serialized graph is noisier: equality rules such as `eq-ref` add reflexive `owl:sameAs` statements for terms that participate in the closure. The user-facing entailment we care about here is still that `ex:alice` is typed as `ex:Human` and `ex:Animal`.

In general, using a graph like this should just look like working with a normal RDFLib graph and should be fairly intuitive.

In [2]:
from rdflib import OWL, RDF, RDFS

graph.bind("ex", "urn:example:")

alice = URIRef("urn:example:alice")
person = URIRef("urn:example:Person")
human = URIRef("urn:example:Human")
animal = URIRef("urn:example:Animal")

print("triples before assertions:", len(graph))

graph.add((person, OWL.equivalentClass, human))
graph.add((human, RDFS.subClassOf, animal))
graph.add((alice, RDF.type, person))

print("triples after assertions:", len(graph))
print("-" * 80)
print(graph.serialize(format="turtle"))

triples before assertions: 0
triples after assertions: 17
--------------------------------------------------------------------------------
@prefix ex: <urn:example:> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

rdf:type owl:sameAs rdf:type .

rdfs:subClassOf owl:sameAs rdfs:subClassOf .

owl:equivalentClass owl:sameAs owl:equivalentClass .

owl:sameAs owl:sameAs owl:sameAs .

ex:alice a ex:Animal,
        ex:Human,
        ex:Person ;
    owl:sameAs ex:alice .

ex:Animal owl:sameAs ex:Animal .

ex:Human rdfs:subClassOf ex:Animal,
        ex:Person ;
    owl:equivalentClass ex:Person ;
    owl:sameAs ex:Human .

ex:Person rdfs:subClassOf ex:Animal,
        ex:Human ;
    owl:equivalentClass ex:Human ;
    owl:sameAs ex:Person .




## 3. Explaining Inferences

To explain one inferred triple, we reconstruct a proof from the derivation log captured during inference.

Here the goal is `(ex:alice rdf:type ex:Animal)`, which should be entailed by typing across `owl:equivalentClass` and then across `rdfs:subClassOf`.
That gives us a `DirectProof` object that we can inspect just enough to see why the inference fired.

In [3]:
from typing import cast

from rdflib_reasoning.engine import (
    DerivationProofReconstructor,
    DirectProof,
    RETEStore,
    TripleFact,
)

graph_engine = cast(RETEStore, dataset.store).engines[graph.identifier]
logger = cast(InMemoryDerivationLogger, graph_engine.derivation_logger)

goal = TripleFact(context=graph.identifier, triple=(alice, RDF.type, animal))
print("Goal triple materialized:", goal.triple in graph)

proof: DirectProof = DerivationProofReconstructor().reconstruct(goal, logger.records)

print("Verdict:", proof.verdict)
print("Top-level node kind:", proof.proof.node_kind)

Goal triple materialized: True
Verdict: proved
Top-level node kind: rule_application


The derivation log includes every recorded rule application for the run, including schema bookkeeping inferences from the OWL 2 RL ruleset.
For this tutorial, the important point is that the reconstructed proof focuses on the rule applications that justify `ex:alice` being typed as `ex:Animal`, rather than every intermediate schema expansion.

In [4]:
print("Recorded derivations:", len(logger.records))

Recorded derivations: 105


### 3.1. Which OWL 2 RL Rules Fired?

The Mermaid rendering is the quickest way to see the class-equivalence and subclass typing steps that justify the final inferred type.

In [5]:
from rdflib_reasoning.engine.proof_notebook import display_proof_mermaid

display_proof_mermaid(proof, namespace_manager=graph.namespace_manager)

```mermaid
flowchart TD
n2[["owl2-rl:cax-sco"]]
n3("(ex:alice rdf:type ex:Animal)")
n3 -->|entailed_by| n2
n4[["owl2-rl:scm-sco"]]
n5("(ex:Person rdfs:subClassOf ex:Animal)")
n5 -->|entailed_by| n4
n6[["owl2-rl:scm-eqc1"]]
n7("(ex:Person rdfs:subClassOf ex:Human)")
n7 -->|entailed_by| n6
n8("(ex:Human rdfs:subClassOf ex:Person)")
n8 -->|entailed_by| n6
n9["(ex:Person owl:equivalentClass ex:Human)"]
n6 -->|had_premise| n9
n4 -->|had_premise| n7
n10["(ex:Human rdfs:subClassOf ex:Animal)"]
n4 -->|had_premise| n10
n2 -->|had_premise| n5
n11["(ex:alice rdf:type ex:Person)"]
n2 -->|had_premise| n11
n1>"(ex:alice rdf:type ex:Animal)"]
n1 -->|justified_by| n3
```

## Notes

- The graph materialization is driven entirely through the RDFLib store integration path via `OWL2_RL_RETE_MATERIALIZED`.
- The proof reconstruction currently uses derivation logs, not a public `graph.explain(...)` API.
- If multiple derivation records can justify the same goal, the current reconstructor chooses the shallowest matching derivation.
- For alternate renderings and raw proof inspection, see [demo-proof-reconstructor.ipynb](./demo-proof-reconstructor.ipynb).
- For contradiction diagnostics when OWL 2 RL false-family rules fire, see [demo-contradiction-rules.ipynb](./demo-contradiction-rules.ipynb).